#### Objective Phase: Time-Series Signal Processing & Feature Architecture

1. **Establish Temporal Data Integrity** (The Ghost Gaps)

Action: Audit the dataset for implicit missingness (dates where stores were closed or data dropped) and construct a continuous Cartesian scaffold to prevent rolling-window calculations from collapsing.

2. **Variance Stabilization & Target Normalization**

Action: Analyze target distribution over time and apply Log1p transformations to compress expanding variance, specifically ensuring the model can handle massive, zero-inflated promotional spikes (e.g., Black Friday markdowns).

3. **Signal Extraction & Stationarity Testing**

Action: Decompose the raw sales signal into its core components (Trend, Seasonality, Residuals) using seasonal decomposition.

Action: Apply the Augmented Dickey-Fuller (ADF) test to the residuals to definitively prove the extracted components successfully captured the non-stationary movement.

4. **Memory Architecture (ACF & PACF Analysis)**

Action: Utilize Autocorrelation and Partial Autocorrelation plots to map the exact "memory" of the sales data, determining the precise historical lag features required without introducing multicollinearity.

5. **Time-Domain Feature Engineering**

Action: Translate the EDA findings into a production-ready feature matrix, including:

Point-in-Time Memory: Strictly grouped Lag features derived from PACF.

Momentum: Safely shifted rolling averages and rolling standard deviations.

Cyclical & Exogenous Encoding: Fourier terms (Sine/Cosine) for calendar continuity and macro-economic indicators (CPI/Unemployment) to capture long-term business cycles.

In [79]:
import polars as pl
import config

In [80]:
lf = pl.scan_parquet(str(config.GOLD_MASTER_PATH))

In [81]:
min_val = lf.select(pl.col("date").min()).collect().item()

#### Implicit Nulls
- Our goal is to find if the data points for a particular weeks are missing, that would make the lagging features we create incorrect.
- Lets say we have two weeks missing data but thats not present so its implicit null, the lagging features would incorrectly use the older data.

**Insights**
- We found **5,447 implicit gaps** (missing records) in the weekly time series
- **Mathematical Corruption:** The pre-engineered `rolling_4_wk_sales_avg` was skipping time, treating data from months or years ago as if it were "last week" because it was technically the previous row.
- **The Fix (Cartesian Scaffolding):** 
     1. We are constructing a **Continuous Date** (Scaffold) from the min/max dates.
     2. We will join our date with store and dept using cross join to get all store + dept + weeks pairs between min and max date.
     3. Then join this scaffold with our main data using left join on the scaffold, this would yield nulls in the new rows created.

- **How to fill the explicit nulls now**
    - This is decided based on the fact that was the store closed on the missing weeks ?. 
    - If yes we would fill with 0 but create a feature for denoting that store was active or inactive i.e mark those weeks as 0 sales (but keep flag is_active=False or store_closed=True).
    - If we dont know the actual reason then we would follow

        1. Build a full calendar grid
            - All (store, dept, week) combinations, even where sales are missing.
        
        2. Impute null weekly sales with 0.0

        3. Compute lags / rolling features on the full grid
            - lag1, rolling_4wk_avg, etc., are computed using calendar‑ordered rows, so the stride is real‑time (1 week, 4 weeks).
        4. Then drop only the “added‑back” rows where sales is null

        5. For training, you keep only rows that were originally observed (no imputed sales), but the lag/rolling features were already computed over the full‑time sequence.

        6. In this flow:
            - The lag/rolling features are correct (they reflect calendar time, not “last‑observed‑row”).
            - The model never sees null sales as targets, so you avoid injecting artificial zeros or imputed values into the label.
        7. This is like:
        Step A (time‑series‑aware): “simulate” the full timeline, compute time‑based features.
        Step B (tabular‑ML): “drop” the extra rows before training, but keep the already‑correct features.


In [82]:
gaps = (lf.select('store', 'dept', 'date', pl.col('date').diff().over(partition_by = ['store', 'dept'])
        .alias("prev_week_diff"))
        .filter((pl.col('prev_week_diff')!=pl.duration(days=7))
        )
        )
                
if gaps.collect().shape[0] > 0:
    print(f"Found {gaps.collect().shape[0]} implicit gaps in the time series!")
    print(gaps.select(["store", "dept", "date", "prev_week_diff"]).head().collect())
    print(gaps.select(["store", "dept", "date", "prev_week_diff"]).tail().collect())
    print(gaps.select(["store", "dept", "date", "prev_week_diff"]).collect().sample(10))
else:
    print("No implicit gaps found. The time series is continuous.")

Found 5447 implicit gaps in the time series!
shape: (5, 4)
┌───────┬──────┬────────────┬────────────────┐
│ store ┆ dept ┆ date       ┆ prev_week_diff │
│ ---   ┆ ---  ┆ ---        ┆ ---            │
│ i32   ┆ i32  ┆ date       ┆ duration[μs]   │
╞═══════╪══════╪════════════╪════════════════╡
│ 1     ┆ 47   ┆ 2010-02-19 ┆ 14d            │
│ 3     ┆ 45   ┆ 2010-02-19 ┆ 14d            │
│ 3     ┆ 51   ┆ 2010-02-19 ┆ 14d            │
│ 7     ┆ 58   ┆ 2010-02-19 ┆ 14d            │
│ 11    ┆ 19   ┆ 2010-02-19 ┆ 14d            │
└───────┴──────┴────────────┴────────────────┘
shape: (5, 4)
┌───────┬──────┬────────────┬────────────────┐
│ store ┆ dept ┆ date       ┆ prev_week_diff │
│ ---   ┆ ---  ┆ ---        ┆ ---            │
│ i32   ┆ i32  ┆ date       ┆ duration[μs]   │
╞═══════╪══════╪════════════╪════════════════╡
│ 44    ┆ 23   ┆ 2012-10-26 ┆ 14d            │
│ 44    ┆ 31   ┆ 2012-10-26 ┆ 14d            │
│ 44    ┆ 33   ┆ 2012-10-26 ┆ 14d            │
│ 44    ┆ 49   ┆ 2012-10-26 ┆ 14d 

In [83]:
prev_week_diff = gaps.select(pl.col('prev_week_diff'))
prev_week_diff.describe()

statistic,prev_week_diff
str,str
"""count""","""5447"""
"""null_count""","""0"""
"""mean""","""42 days, 13:19:26.513677"""
"""std""",null
"""min""","""14 days, 0:00:00"""
"""25%""","""14 days, 0:00:00"""
"""50%""","""21 days, 0:00:00"""
"""75%""","""42 days, 0:00:00"""
"""max""","""882 days, 0:00:00"""


In [84]:
gaps.filter(pl.col('prev_week_diff')== pl.duration(days=882)).collect()

store,dept,date,prev_week_diff
i32,i32,date,duration[μs]
35,78,2012-09-07,882d


In [85]:
lf.filter(pl.col('store') == 35, pl.col('dept') == 78).collect()

store,dept,date,weekly_sales,isholiday,store_type,store_size,temperature,fuel_price,cpi,unemployment,markdown1,markdown2,markdown3,markdown4,markdown5,total_markdown,month,week_of_year,rolling_4_wk_sales_avg,sales_last_year,cpi_lag_3_month
i32,i32,date,f64,bool,str,i32,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,f64,f64,f32
35,78,2010-02-05,31.0,false,"""B""",103681,27.19,2.784,135.352463,9.262,0.0,0.0,0.0,0.0,0.0,0.0,2,5,null,null,135.352463
35,78,2010-02-12,5.0,true,"""B""",103681,29.81,2.773,135.411301,9.262,0.0,0.0,0.0,0.0,0.0,0.0,2,6,31.0,null,135.411301
35,78,2010-04-02,12.0,false,"""B""",103681,46.9,2.85,135.746506,9.051,0.0,0.0,0.0,0.0,0.0,0.0,4,13,18.0,null,135.746506
35,78,2010-04-09,12.0,false,"""B""",103681,62.62,2.869,135.785629,9.051,0.0,0.0,0.0,0.0,0.0,0.0,4,14,16.0,null,135.785629
35,78,2012-09-07,3.0,true,"""B""",103681,76.0,3.911,142.500305,8.839,11678.320312,0.0,53.349998,1976.76001,8415.889648,22124.320312,9,36,15.0,null,142.500305


In [86]:
import polars as pl

# 1. Get the global date range
date_range = pl.date_range(
    start=lf.select(pl.col("date").min()).collect().item(),
    end=lf.select(pl.col("date").max()).collect().item(),
    interval="1w",
    eager=True
).alias("date").to_frame()

# 2. Get unique Store/Dept combinations
unique_keys = lf.select(["store", "dept"]).unique().collect()

# 3. Build the Scaffold (Every Store/Dept has every Date)
scaffold = unique_keys.join(date_range, how="cross")

# 4. Left join to expose the gaps (they will have nulls)
df_grid = scaffold.join(lf.collect(), on=["store", "dept", "date"], how="left")

# 5. THE CRUCIAL FLAG
# Tag the rows before we fill the nulls
df_grid = df_grid.with_columns(
    pl.col("weekly_sales").is_not_null().alias("is_original_row")
)

# 6: Impute the nulls with 0.0 to act as our mathematical bridge
df_grid = df_grid.with_columns(
    pl.col("weekly_sales").fill_null(0.0).alias("weekly_sales_bridged")
)

# 7: Compute lags and rolling features on the continuous, 0-filled grid
df_features = df_grid.with_columns([
    # Lag 1 based on the bridged data
    pl.col("weekly_sales_bridged").shift(1).over(["store", "dept"]).alias("sales_lag_1"),
    
    # 4-week rolling average based on the bridged data
    pl.col("weekly_sales_bridged").shift(1).rolling_mean(window_size=4).over(["store", "dept"]).alias("rolling_4wk_avg")
])

# 8: Drop the "added-back" rows using our flag
# This returns the dataset to its original physical rows, but now carrying perfect calendar features
df_final_training = df_features.filter(pl.col("is_original_row") == True)

# Cleanup: Drop the bridging column and flag so XGBoost doesn't get confused
df_final_training = df_final_training.drop(["weekly_sales_bridged", "is_original_row"])

print(f"Original Row Count: {lf.collect().height}")
print(f"Scaffolded Row Count: {df_grid.height}")
print(f"Final model ready dataset Row Count: {df_final_training.height}")


Original Row Count: 421570
Scaffolded Row Count: 476333
Final model ready dataset Row Count: 421570
